[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/03_lasso_regression/Lasso_Regression.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/03_lasso_regression/Lasso_Regression.ipynb)

# Notebook 03 — Lasso Regression
## Automatic Feature Selection via Sparsity

**Dataset**: California Housing
**Question**: *Which features actually matter — and can some be dropped entirely?*
**Type**: Supervised Regression with L1 Regularisation

---

## Section 1 — What Is Lasso Regression?

### Very similar to Ridge — one key difference

Ridge adds `α × Σ(w²)` — this shrinks weights toward zero but never exactly to zero.

Lasso adds `α × Σ(|w|)` — using **absolute value** instead of squared.

This small mathematical change has a big practical effect:
- **Ridge**: every feature keeps a non-zero weight (just small)
- **Lasso**: unimportant features get their weight driven to exactly **zero**

Features with zero weight are completely ignored — **Lasso selects features automatically**.

> **Analogy**: Ridge is like turning down the volume on all instruments in a band.
> Lasso is like muting the instruments that are not contributing to the song.

### When to use Lasso over Ridge

| Situation | Prefer |
|---|---|
| Many features, suspect most are irrelevant | **Lasso** (zeros them out) |
| All features probably contribute | **Ridge** (shrinks all evenly) |
| Want an interpretable reduced model | **Lasso** |
| High correlation between features | **Ridge** (handles multicollinearity better) |

## Section 2 — How Does It Learn?

The gradient of the L1 penalty has a **discontinuity at zero** — unlike the smooth L2 gradient.

When a weight is small, the gradient from `|w|` is constant (either +α or −α).
This constant force is strong enough to pull small weights all the way to zero and keep them there.
With `w²`, the force shrinks as the weight shrinks — never quite reaching zero.

The optimisation algorithm for Lasso is called **Coordinate Descent**:
- Update one weight at a time (not all simultaneously)
- Apply a "soft threshold" that zeros out weights smaller than α

> This is why Lasso is also called a **sparse** model — its solution has many zeros.

## Section 3 — What Does the Data Need?

### Gradient-based models need scaling

All three (Linear, Ridge, Lasso) use gradient descent — the same scaling argument applies:
features on different scales cause unequal gradient steps.

| Feature | Description | Transform |
|---|---|---|
| `MedInc` | Median income (in $10k) | StandardScaler |
| `HouseAge` | Median house age (years) | StandardScaler |
| `AveRooms` | Average rooms per household | StandardScaler + clip outliers |
| `AveBedrms` | Average bedrooms per household | StandardScaler + clip outliers |
| `Population` | Block group population | StandardScaler |
| `AveOccup` | Average household occupancy | StandardScaler + clip outliers |
| `Latitude` | Geographic latitude | StandardScaler |
| `Longitude` | Geographic longitude | StandardScaler |

> **Split first, then scale** — fit the scaler on training data only to avoid data leakage.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target
for col in ['AveRooms', 'AveBedrms', 'AveOccup']:
    df[col] = df[col].clip(upper=df[col].quantile(0.99))

FEATURES = housing.feature_names.tolist()
TARGET   = 'MedHouseVal'

print(f"California Housing: {df.shape[0]} districts, {len(FEATURES)} features")
print(f"Target (MedHouseVal): median house value in $100k, range {df[TARGET].min():.2f}–{df[TARGET].max():.2f}")
df.head(6)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_raw = df[FEATURES].values
y     = df[TARGET].values

X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr_raw)
X_test  = scaler.transform(X_te_raw)

stats = pd.DataFrame({
    'Feature'     : FEATURES,
    'Raw mean'    : X_tr_raw.mean(axis=0).round(3),
    'Raw std'     : X_tr_raw.std(axis=0).round(3),
    'Scaled mean' : X_train.mean(axis=0).round(4),
    'Scaled std'  : X_train.std(axis=0).round(4),
})
print(f"Train: {len(X_train)} districts  |  Test: {len(X_test)} districts")
print()
print(stats.to_string(index=False))

## Section 4 — Train the Model and Read the Rules

In [ ]:
from sklearn.linear_model import Lasso, LassoCV

# Best alpha via cross-validation
lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)
lasso_cv.fit(X_train, y_train)
best_alpha = lasso_cv.alpha_
y_pred = lasso_cv.predict(X_test)

from sklearn.metrics import mean_squared_error, r2_score
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

n_zero    = (lasso_cv.coef_ == 0).sum()
n_nonzero = (lasso_cv.coef_ != 0).sum()

print(f"Best alpha (5-fold CV): {best_alpha:.6f}")
print(f"RMSE: {rmse:.4f}  |  R²: {r2:.4f}")
print(f"Features used  : {n_nonzero} / {len(FEATURES)}")
print(f"Features zeroed: {n_zero}")
print()
coef_df = pd.DataFrame({'Feature': FEATURES, 'Weight': lasso_cv.coef_.round(4)})
print(coef_df.to_string(index=False))

In [ ]:
# Coefficient path — watch features drop to zero as alpha increases
alphas_path = np.logspace(-3, 1, 300)
coef_paths  = []
for a in alphas_path:
    m = Lasso(alpha=a, max_iter=10000)
    m.fit(X_train, y_train)
    coef_paths.append(m.coef_.copy())
coef_paths = np.array(coef_paths)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

for i, feat in enumerate(FEATURES):
    ax1.plot(np.log10(alphas_path), coef_paths[:, i], lw=1.5, label=feat)
ax1.axhline(0, color='black', lw=0.8, ls='--')
ax1.axvline(np.log10(best_alpha), color='grey', lw=1, ls=':', label=f'Best alpha={best_alpha:.4f}')
ax1.set_xlabel('log10(alpha)  — regularisation strength →', fontsize=11)
ax1.set_ylabel('Feature weight', fontsize=11)
ax1.set_title('Lasso: Coefficient Path\n(weights hit exactly 0 and stay there)', fontsize=12)
ax1.legend(fontsize=8, ncol=2)

n_nonzero_path = (coef_paths != 0).sum(axis=1)
ax2.plot(np.log10(alphas_path), n_nonzero_path, 'steelblue', lw=2)
ax2.axvline(np.log10(best_alpha), color='grey', lw=1, ls=':')
ax2.set_xlabel('log10(alpha)', fontsize=11)
ax2.set_ylabel('Number of non-zero features', fontsize=11)
ax2.set_title('Features Surviving vs Alpha\n(Lasso selects features automatically)', fontsize=12)

plt.tight_layout()
plt.show()
print('Key difference from Ridge: weights drop to exactly 0, not just close to 0.')